### Transform drivers Data

In [0]:
%run ../00-common/01-environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

#### 1. Read drivers data

In [0]:
drivers_df = spark.read.table(bronze_table)
display(drivers_df)

#### 2. `Check` and remove NUll values

In [0]:
from pyspark.sql import functions as F

null_counts_df = drivers_df.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in drivers_df.columns
])

display(null_counts_df)

In [0]:
display(
    drivers_df.filter(
        (F.col("nationality").isNull()) | (F.col("dateOfBirth").isNull())
    )
)

We'll keep them, because these are personal atrributes and are not too many records 

#### 3. Drop Unncessesory columns

In [0]:
drivers_selected_df = drivers_df.drop("url")
display(drivers_selected_df)

#### 4. Rename Columns

In [0]:
drivers_renamed_df = drivers_selected_df.withColumnsRenamed({
    "driverId": "driver_id",
    "dateOfBirth": "date_of_birth",
})

#### 5. Check for duplicates and Remove if needed

In [0]:
drivers_distinct_df = drivers_renamed_df.distinct()
drivers_distinct_df.display()

#### Concatinate givenName and familyName to create full_name and Apply Title Case 

In [0]:
drivers_concatinated_df = drivers_distinct_df.withColumn(
    "driver_name",
    F.initcap(
        F.concat_ws(
            " ",
            F.col("name")["givenName"],
            F.col("name")["familyName"]
        )
    )
).drop("name")

display(drivers_concatinated_df)

### 6. Transform Values of Columns nationality, name to Title Case

In [0]:
drivers_final_df = drivers_concatinated_df.withColumns({
    "nationality": F.initcap(F.col("nationality"))
})

display(drivers_final_df)

### 7. Writing data to bronze table

In [0]:
(
    drivers_final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))